# Building a sample dataset for testing derivative products across different Landsat sensors

This notebook is designed to create a sample set for testing derivative products across the continent, where the testing needs to compare data derived from multiple landsat sensors. It is designed to do the following:

- Import a geojson containing the footprints of the 'golden tiles' that have been previously used for validating GeoMAD products. These tiles are distributed across Australia and across a range of environment types.
- Load a datacube dataset for the selected sensors (in this case, Landsat 7, and Landsat 8/9), using one or more of the golden tiles as the geometry for the datacube query
- Compare the resulting datasets and find the timesteps that overlap ( +/- a timeframe set by the user, e.g. 48 hours)
- If the resulting filtered dataset is very large, create a subset based on randomly selecting smaller regions or pixels
- Export the resulting geojson so it can be used as an input to test workflows

In [1]:
!pip uninstall dea-tools -y

In [2]:
import os
import glob
import datacube
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
import pprint
from datetime import timedelta
import matplotlib.pyplot as plt
from datacube.utils.geometry import CRS, Geometry, GeoBox
from datacube.utils import masking
from pathlib import Path
from shapely.geometry import box

import pystac_client
import planetary_computer

import odc.geo.xr
from odc.geo.xr import assign_crs
from odc.io.cgroups import get_cpu_quota
from odc.geo.geom import BoundingBox

import sys

sys.path.insert(1, "../../../Tools")
from dea_tools.datahandling import load_ard
from dea_tools.classification import collect_training_data, HiddenPrints
from dea_tools.dask import create_local_dask_cluster
from dea_tools.plotting import rgb, display_map
from dea_tools.spatial import xr_vectorize, xr_rasterize
from dea_tools.bandindices import calculate_indices

import warnings

warnings.filterwarnings("ignore")


In [3]:
client = create_local_dask_cluster(return_client=True)


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/8787/status,
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/8787/status,Workers: 1
Total threads: 15,Total memory: 114.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44273,Workers: 1
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/8787/status,Total threads: 15
Started: Just now,Total memory: 114.00 GiB
Comm: tcp://127.0.0.1:37789,Total threads: 15
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/42847/status,Memory: 114.00 GiB
Nanny: tcp://127.0.0.1:44709,


In [4]:
# funtion from Chad to sample ABARES landuse data

def random_sampling(
    da, n, sampling="stratified_random", manual_class_ratios=None, out_fname=None
):
    """
    Creates randomly sampled points for post-classification
    accuracy assessment.

    Params:
    -------
    da: xarray.DataArray
        A classified 2-dimensional xarray.DataArray
    n: int
        Total number of points to sample. Ignored if providing
        a dictionary of {class:numofpoints} to 'manual_class_ratios'
    sampling: str
        'stratified_random' = Create points that are randomly
        distributed within each class, where each class has a
        number of points proportional to its relative area.
        'equal_stratified_random' = Create points that are randomly
        distributed within each class, where each class has the
        same number of points.
        'random' = Create points that are randomly distributed
        throughout the image.
        'manual' = user definined, each class is allocated a
        specified number of points, supply a manual_class_ratio
        dictionary mapping number of points to each class
    manual_class_ratios: dict
        If setting sampling to 'manual', the provide a dictionary
        of type {'class': numofpoints} mapping the number of points
        to generate for each class.
    out_fname: str
        If providing a filepath name, e.g 'sample_points.shp', the
        function will export a shapefile/geojson of the sampling
        points to file.

    Output
    ------
    GeoPandas.Dataframe

    """

    if sampling not in [
        "stratified_random",
        "equal_stratified_random",
        "random",
        "manual",
    ]:
        raise ValueError(
            "Sampling strategy must be one of 'stratified_random', "
            + "'equal_stratified_random', 'random', or 'manual'"
        )

    # open the dataset as a pandas dataframe
    da = da.squeeze()
    df = da.to_dataframe() #name="class"
    df = df.dropna(how="any")

    # list to store points
    samples = []

    if sampling == "stratified_random":
        # determine class ratios in image
        class_ratio = pd.DataFrame(
            {
                "proportion": df["class"].value_counts(normalize=True),
                "class": df["class"].value_counts(normalize=True).keys(),
            }
        )

        for _class in class_ratio["class"]:
            # use relative proportions of classes to sample df
            no_of_points = (
                n * class_ratio[class_ratio["class"] == _class]["proportion"].values[0]
            )
            # random sample each class
            print(
                "Class "
                + str(_class)
                + ": sampling at "
                + str(round(no_of_points))
                + " coordinates"
            )
            sample_loc = df[df["class"] == _class].sample(n=int(round(no_of_points)))
            samples.append(sample_loc)

    if sampling == "equal_stratified_random":
        classes = np.unique(df["class"])

        for _class in classes:
            # use relative proportions of classes to sample df
            no_of_points = n / len(classes)
            # random sample each classes
            try:
                sample_loc = df[df["class"] == _class].sample(
                    n=int(round(no_of_points))
                )
                print(
                    "Class "
                    + str(_class)
                    + ": sampling at "
                    + str(round(no_of_points))
                    + " coordinates"
                )
                samples.append(sample_loc)

            except ValueError:
                print(
                    "Requested more sample points than population of pixels for class "
                    + str(_class)
                    + ", skipping"
                )
                pass

    if sampling == "random":
        no_of_points = n
        # random sample entire df
        print(
            "Randomly sampling dataAraay at "
            + str(round(no_of_points))
            + " coordinates"
        )
        sample_loc = df.dropna().sample(n=int(round(no_of_points)))
        samples.append(sample_loc)

    if sampling == "manual":
        if isinstance(manual_class_ratios, dict):
            # check classes in dict match classes in data
            classes = np.unique(df["class"])
            dict_classes = list(manual_class_ratios.keys())

            if set(dict_classes).issubset([str(i) for i in classes]):
                # mask for just those classes in the provided dictionary
                mask = np.isin(classes, np.array(dict_classes).astype(type(classes[0])))
                classes = classes[mask]
                # run sampling
                for _class in classes:
                    no_of_points = manual_class_ratios.get(str(_class))
                    # random sample each class
                    try:
                        sample_loc = df[df["class"] == _class].sample(
                            n=int(round(no_of_points))
                        )
                        print(
                            "Class "
                            + str(_class)
                            + ": sampled at "
                            + str(round(no_of_points))
                            + " coordinates"
                        )
                        samples.append(sample_loc)

                    except ValueError:
                        print(
                            "Requested more sample points than population of pixels for class "
                            + str(_class)
                            + ", skipping"
                        )
                        pass

            else:
                raise ValueError(
                    "Some or all of the classes in 'manual_class_ratio' dictionary do not"
                    + " match the classes in the supplied dataArray. "
                    + "DataArray classes: "
                    + str(classes)
                    + ", Supplied dict classes: "
                    + str(list(manual_class_ratios.keys()))
                )

        else:
            raise ValueError(
                "Must supply a dictionary mapping {'class': numofpoints} if sampling"
                + " is set to 'manual'"
            )

    # join back into single datafame
    all_samples = pd.concat([samples[i] for i in range(0, len(samples))])

    # get pd.mulitindex coords as list
    y = [i[0] for i in list(all_samples.index)]
    x = [i[1] for i in list(all_samples.index)]

    # create geopandas dataframe
    gdf = gpd.GeoDataFrame(
        all_samples, crs=f"EPSG:{da.odc.crs.epsg}", geometry=gpd.points_from_xy(x, y)
    ).reset_index()

    gdf = gdf.drop(["x", "y"], axis=1)

    if out_fname is not None:
        gdf.to_file(out_fname)

    return gdf


In [5]:
def get_abares_classes(abares_year='2020', sample_size=1000):
    #There are 2 years of ABARES CLUM data in the datacube, select one.
    if abares_year =='2020':
        product = "abares_clum_2020"
    elif abares_year == '2023':
        product = "abares_clum_2023"
    else:
        print("Please enter valid ABARES CLUM year: 2020 or 2023.")

    query_abares = {
        "resolution": resolution,
        "output_crs": output_crs,
        "group_by": "solar_day",
        "product": product,
        "resolution": (-30, 30),
        "dask_chunks": {"time":1, "x":2048, "y": 2048}
    }

    #modify the abares query to select a golden tile. This will be removed at a later date.
    #query_abares = select_tile(test_tiles_gdf, region_codes, query_abares)

    ds_clum = dc.load(**query_abares)

    ds_clum_classes = (
        ds_clum // 100
    ) * 100  # convert all the classes to only have the parent 6 classes for now.

    ds_output = ds_clum_classes.alum_class
    ds_output = ds_output.squeeze().drop_vars("time")

    return ds_output

In [6]:
def feature_layers(query):
    dc = datacube.Datacube()
    ds = dc.load(product='ga_ls8cls9c_gm_cyear_3',
                measurements=[
                "nbart_blue",
                "nbart_green",
                "nbart_red",
                "nbart_nir",
                "nbart_swir_1",
                "nbart_swir_2"],
                **query)

    
    ds = calculate_indices(
        ds, index=["TCW", "TCG", "TCB", "TCW_DEA", "TCB_DEA", "TCG_DEA", "TCW_ls8", "TCB_ls8", "TCG_ls8"], drop=False, collection="ga_ls_3"
    )

    return ds


### Analysis parameters


In [7]:
output_dir = "temp_outputs"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [8]:
output_crs = "EPSG:3577"
time = ('2020-01', '2020-12')

au_boundary_dir = "australia_100km_buffer.geojson"

In [9]:
au_boundary_gpd = gpd.read_file(au_boundary_dir)

xmin, ymin, xmax, ymax = au_boundary_gpd.total_bounds

# x = (122.10, 122.48)
# y = (-17.91, -18.28)

x = (111, 154)
y = (-8, -45)

bbox = BoundingBox.from_xy(x, y)
time_range = "/".join(time)


print(f"xmin: {xmin}, xmax:{xmax}, ymin: {ymin}, ymax:{ymax}")
print(au_boundary_gpd.total_bounds)

xmin: 111.9154159687281, xmax:154.67347079958884, ymin: -44.79649874840166, ymax:-8.20633892690521
[111.91541597 -44.79649875 154.6734708   -8.20633893]


In [10]:
# Try using ESA worldvoer instead of ABARES

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)


In [11]:

# Search for STAC items from "esa-worldcover" product
search = catalog.search(
    collections=["esa-worldcover"],
    bbox=bbox,
    datetime = time_range
)

# Check how many items were returned
items = list(search.get_items())

In [12]:
ds_worldcover = odc.stac.load(
    items,
    bbox=bbox,
    bands=['map'],
    crs = 'EPSG:3577',
    resolution=120,
    chunks={'x': 2048, 'y':2048}
)


In [13]:
ds_worldcover = ds_worldcover.squeeze().drop_vars('time')
ds_worldcover = ds_worldcover.rename({'map': 'class'})

In [14]:
# to keep collect_training-Data runs small to avoid S3 errors, do each class individually
# fname_outpath_worldcover_samples = f"{output_dir}/worldcover_stratified_sample_points_2020.gpkg"

worldcover_sample = random_sampling(
    ds_worldcover, 
    4000, 
    sampling="equal_stratified_random", 
    #out_fname=fname_outpath_worldcover_samples
)

/env/lib/python3.10/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(
/env/lib/python3.10/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


Class 0: sampling at 400 coordinates
Class 10: sampling at 400 coordinates
Class 20: sampling at 400 coordinates
Class 30: sampling at 400 coordinates
Class 40: sampling at 400 coordinates
Class 50: sampling at 400 coordinates
Class 60: sampling at 400 coordinates
Class 80: sampling at 400 coordinates
Class 90: sampling at 400 coordinates
Class 95: sampling at 400 coordinates


In [15]:
worldcover_class_list = list(worldcover_sample['class'].unique())

In [16]:
worldcover_class_list

[0, 10, 20, 30, 40, 50, 60, 80, 90, 95]

In [17]:
worldcover_class_dict={10:'tree_cover',
                      20: 'shrubland',
                      30: 'grassland',
                      40: 'cropland',
                      50: 'built-up',
                      60: 'bare_sparse_veg',
                      70: 'snow_ice',
                      80: 'water_permanent',
                      90: 'wetland_herbaceous',
                      95: 'mangroves'}

In [18]:
# save out each ESA worldcover class samples as individual geopackages to then collect sample data

for worldcover_class in worldcover_class_list:
    if worldcover_class not in worldcover_class_dict:
        continue
    class_name = worldcover_class_dict[worldcover_class]
    print(class_name)
    fname_outpath_worldcover_samples = f"{output_dir}/worldcover_class_{class_name}_stratified_sample_points_2020.gpkg"
    samples = worldcover_sample.loc[worldcover_sample['class']==worldcover_class]
    samples.to_file(fname_outpath_worldcover_samples, drover='gpkg')
    

tree_cover
shrubland
grassland
cropland
built-up
bare_sparse_veg
water_permanent
wetland_herbaceous
mangroves


## Set up query and function for collecting data from datacube, stop dask client

- the `collect_sample_data` function doesn't play nicely with a local dask cluster, so the local dask client needs to be shut down before running that cell.

In [19]:
ncpus = round(get_cpu_quota())
ncpus

15

In [20]:
# read the geopackage file back in to then collect sample data from datacube for TC's

query = {
    "time": ('2020'),
    "resolution": (-120,120),
    "output_crs": output_crs
}

In [21]:
client.shutdown()

In [22]:

for fname in glob.glob(f"{output_dir}/*.gpkg"):
    print(fname)

    root, _ = os.path.splitext(fname)
    new_fname = root + '.csv'

    print(new_fname)
    
    samples = gpd.read_file(fname)
    
    column_names, model_input = collect_training_data(
        gdf=samples,
        dc_query=query,
        ncpus=ncpus,
        return_coords=True,
        field="class",
        #zonal_stats="mean",  # not actually going to use this but it complains if set to False.
        feature_func=feature_layers,
    )

    print(column_names)
    print(model_input)
    #np.savetxt(new_fname, model_input, header=" ".join(column_names), fmt="%4f")
    model_df = pd.DataFrame(model_input)
    model_df.to_csv(new_fname, index=False, header=column_names, float_format='%4f')


temp_outputs/worldcover_class_cropland_stratified_sample_points_2020.gpkg
temp_outputs/worldcover_class_cropland_stratified_sample_points_2020.csv


  0%|          | 0/400 [00:00<?, ?it/s]

Percentage of possible fails after run 1 = 0.0 %
Removed 0 rows wth NaNs &/or Infs
Output shape:  (398, 18)
['class', 'nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir', 'nbart_swir_1', 'nbart_swir_2', 'TCW', 'TCG', 'TCB', 'TCW_DEA', 'TCB_DEA', 'TCG_DEA', 'TCW_ls8', 'TCB_ls8', 'TCG_ls8', 'x_coord', 'y_coord']
[[ 4.000000e+01  9.410000e+02  1.218000e+03 ...  7.040190e-03
  -1.189980e+06 -3.804900e+06]
 [ 4.000000e+01  8.140000e+02  1.123000e+03 ...  4.641256e-02
  -1.196100e+06 -3.805260e+06]
 [ 4.000000e+01  7.410000e+02  1.083000e+03 ...  3.234856e-02
  -1.637580e+06 -3.184620e+06]
 ...
 [ 4.000000e+01  6.220000e+02  9.490000e+02 ...  9.237667e-02
  -1.161540e+06 -3.631020e+06]
 [ 4.000000e+01  6.810000e+02  1.042000e+03 ...  6.670376e-02
   8.981400e+05 -3.953820e+06]
 [ 4.000000e+01  4.770000e+02  6.910000e+02 ...  1.144133e-02
   1.682100e+06 -2.782500e+06]]
temp_outputs/worldcover_class_water_permanent_stratified_sample_points_2020.gpkg
temp_outputs/worldcover_class_water_perma

  0%|          | 0/400 [00:00<?, ?it/s]

Percentage of possible fails after run 1 = 0.0 %
Removed 0 rows wth NaNs &/or Infs
Output shape:  (300, 18)
['class', 'nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir', 'nbart_swir_1', 'nbart_swir_2', 'TCW', 'TCG', 'TCB', 'TCW_DEA', 'TCB_DEA', 'TCG_DEA', 'TCW_ls8', 'TCB_ls8', 'TCG_ls8', 'x_coord', 'y_coord']
[[ 8.000000e+01  3.480000e+02  2.390000e+02 ... -1.690560e-02
   1.600260e+06 -4.171500e+06]
 [ 8.000000e+01  3.260000e+02  2.190000e+02 ... -1.606204e-02
   1.605420e+06 -4.150260e+06]
 [ 8.000000e+01  4.940000e+02  3.380000e+02 ... -2.224478e-02
   8.052600e+05 -1.740420e+06]
 ...
 [ 8.000000e+01  8.060000e+02  6.400000e+02 ... -4.435013e-02
  -6.394200e+05 -1.424700e+06]
 [ 8.000000e+01  3.110000e+02  1.750000e+02 ... -1.450304e-02
  -3.371400e+05 -3.496980e+06]
 [ 8.000000e+01  4.900000e+02  4.590000e+02 ... -2.671458e-02
   2.177400e+05 -1.231260e+06]]
temp_outputs/worldcover_class_shrubland_stratified_sample_points_2020.gpkg
temp_outputs/worldcover_class_shrubland_stratif

...

Percentage of possible fails after run 1 = 0.0 %
Removed 0 rows wth NaNs &/or Infs
Output shape:  (396, 18)
['class', 'nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir', 'nbart_swir_1', 'nbart_swir_2', 'TCW', 'TCG', 'TCB', 'TCW_DEA', 'TCB_DEA', 'TCG_DEA', 'TCW_ls8', 'TCB_ls8', 'TCG_ls8', 'x_coord', 'y_coord']
[[ 9.000000e+01  8.060000e+02  1.004000e+03 ...  3.260994e-02
  -1.013340e+06 -1.960860e+06]
 [ 9.000000e+01  1.642000e+03  2.283000e+03 ... -2.995913e-02
   1.009740e+06 -3.180300e+06]
 [ 9.000000e+01  9.950000e+02  1.725000e+03 ...  3.740800e-04
   4.855800e+05 -1.683420e+06]
 ...
 [ 9.000000e+01  3.680000e+02  4.690000e+02 ...  6.970073e-02
   1.233780e+06 -1.320420e+06]
 [ 9.000000e+01  4.360000e+02  6.040000e+02 ...  5.301272e-02
   1.098300e+06 -1.330620e+06]
 [ 9.000000e+01  5.140000e+02  7.230000e+02 ...  7.083959e-02
   1.941180e+06 -3.604020e+06]]
temp_outputs/worldcover_class_built-up_stratified_sample_points_2020.gpkg
temp_outputs/worldcover_class_built-up_stratifie

  0%|          | 0/400 [00:00<?, ?it/s]

Percentage of possible fails after run 1 = 0.0 %
Removed 0 rows wth NaNs &/or Infs
Output shape:  (393, 18)
['class', 'nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir', 'nbart_swir_1', 'nbart_swir_2', 'TCW', 'TCG', 'TCB', 'TCW_DEA', 'TCB_DEA', 'TCG_DEA', 'TCW_ls8', 'TCB_ls8', 'TCG_ls8', 'x_coord', 'y_coord']
[[ 5.000000e+01  7.340000e+02  1.008000e+03 ...  6.113685e-02
   2.044620e+06 -3.120900e+06]
 [ 5.000000e+01  1.454000e+03  1.723000e+03 ... -5.351253e-02
   2.068500e+06 -3.204660e+06]
 [ 5.000000e+01  1.174000e+03  1.390000e+03 ...  1.722656e-02
   5.890200e+05 -3.856860e+06]
 ...
 [ 5.000000e+01  1.425000e+03  1.721000e+03 ...  3.014314e-02
   2.062620e+06 -2.904300e+06]
 [ 5.000000e+01  9.480000e+02  1.153000e+03 ...  9.477589e-02
   1.460700e+06 -1.870380e+06]
 [ 5.000000e+01  6.340000e+02  8.860000e+02 ...  6.490888e-02
   1.850340e+06 -2.684700e+06]]
temp_outputs/worldcover_class_grassland_stratified_sample_points_2020.gpkg
temp_outputs/worldcover_class_grassland_stratif

  0%|          | 0/400 [00:00<?, ?it/s]

Percentage of possible fails after run 1 = 0.0 %
Removed 0 rows wth NaNs &/or Infs
Output shape:  (397, 18)
['class', 'nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir', 'nbart_swir_1', 'nbart_swir_2', 'TCW', 'TCG', 'TCB', 'TCW_DEA', 'TCB_DEA', 'TCG_DEA', 'TCW_ls8', 'TCB_ls8', 'TCG_ls8', 'x_coord', 'y_coord']
[[ 3.000000e+01  5.490000e+02  9.960000e+02 ...  2.775211e-02
  -1.257300e+06 -2.975940e+06]
 [ 3.000000e+01  5.540000e+02  9.750000e+02 ...  1.407405e-02
  -1.469460e+06 -2.917140e+06]
 [ 3.000000e+01  6.240000e+02  1.176000e+03 ...  2.608015e-02
   3.989400e+05 -2.929260e+06]
 ...
 [ 3.000000e+01  5.400000e+02  7.920000e+02 ...  2.636235e-02
  -2.037000e+05 -1.862820e+06]
 [ 3.000000e+01  6.620000e+02  1.215000e+03 ...  7.572500e-03
   7.143000e+05 -2.371980e+06]
 [ 3.000000e+01  6.110000e+02  1.237000e+03 ...  3.512735e-02
   8.685000e+05 -3.156180e+06]]


In [23]:

# for fname in glob.glob(f"{output_dir}/*.gpkg"):
#     print(fname)

#     root, _ = os.path.splitext(fname)
#     new_fname = root + '.csv'

#     print(new_fname)
    
#     samples = gpd.read_file(fname)
    
#     column_names, model_input = collect_training_data(
#         gdf=samples,
#         dc_query=query,
#         ncpus=ncpus,
#         return_coords=True,
#         field="class",
#         #zonal_stats="mean",  # not actually going to use this but it complains if set to False.
#         feature_func=feature_layers,
#     )

#     np.savetxt(new_fname, model_input, header=" ".join(column_names), fmt="%4f")